# فاز دوم — نوت‌بوک ارائه (Demo + Report)

این نوت‌بوک خروجی‌های فاز دوم را از `results/` می‌خواند و یک جمع‌بندی قابل ارائه تولید می‌کند.


In [ ]:
from pathlib import Path

# این نوت‌بوک طوری نوشته شده که هم اگر از ریشه‌ی پروژه اجرا شود کار کند
# و هم اگر داخل پوشه‌ی notebooks باز/اجرا شود.
_here = Path('.').resolve()
if (_here / 'results').exists():
    ROOT = _here
elif (_here.parent / 'results').exists():
    ROOT = _here.parent
else:
    ROOT = _here

RESULTS = ROOT / 'results' / 'tables'
FIGS = ROOT / 'figures'

print("ROOT =", ROOT)
print("RESULTS =", RESULTS)
print("FIGS =", FIGS)


## 1) مقایسه مدل‌ها (Baseline Comparison)


In [ ]:
df_cmp = pd.read_csv(paths['model_comparison'])
df_cmp.sort_values(by=['valid_f1','valid_auc'], ascending=False)


## 2) نتایج دقیق‌تر هر مدل (valid/test)


In [ ]:
rep = json.loads(paths['model_reports'].read_text(encoding='utf-8'))
rows = []
for m, rr in rep['reports'].items():
    rows.append({
        'model': m,
        'valid_f1': rr['valid']['f1'],
        'valid_auc': rr['valid']['roc_auc'],
        'test_f1': rr['test']['f1'],
        'test_auc': rr['test']['roc_auc'],
        'test_acc': rr['test']['accuracy'],
    })
pd.DataFrame(rows).sort_values(['valid_f1','valid_auc'], ascending=False)


## 3) Hyperparameter Tuning


In [ ]:
df_tune = pd.read_csv(paths['tuning_results'])
df_tune.sort_values('cv_best_score', ascending=False).head(10)


In [ ]:
best = json.loads(paths['best_model_report'].read_text(encoding='utf-8'))
best['best_model'], best['best_cv_score']


In [ ]:
best['best_params']


### عملکرد Best Model روی Valid/Test (thr=0.5)


In [ ]:
best['valid']


In [ ]:
best['test']


## 4) Threshold Tuning + Calibration


In [ ]:
thr = json.loads(paths['threshold_summary'].read_text(encoding='utf-8'))
thr['calibration'], thr['best_threshold_valid']


In [ ]:
thr['test_at_0.5']


In [ ]:
thr['test_at_best_threshold']


## 5) نمایش نمودارهای کلیدی (اگر موجود باشند)


### گالری تصاویر (Phase-2 و SVM-RBF)

اگر فقط می‌خواهید خروجی‌ها را سریع ببینید، همه‌ی نمودارها داخل پوشه‌ی `figures/` هستند.
در سلول بعدی، چند نمودار کلیدی را یک‌جا نمایش می‌دهم (SVM-RBF و خروجی‌های Phase-2 مثل Calibration، PR و انتخاب آستانه).

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

def show_if_exists(p, title=None, figsize=(7,5)):
    p = Path(p)
    if p.exists():
        img = plt.imread(p)
        plt.figure(figsize=figsize)
        plt.imshow(img)
        plt.axis('off')
        if title:
            plt.title(title)
        plt.show()
    else:
        print(f"[MISSING] {p}")

# چند نمودار کلیدی که معمولاً برای گزارش سریع نگاه می‌کنم
key_figs = [
    (FIGS/'SVM-RBF_cm_test.png', 'SVM-RBF — CM (Test)'),
    (FIGS/'SVM-RBF_roc_test.png', 'SVM-RBF — ROC (Test)'),
    (FIGS/'RandomForest_feature_importance_top20.png', 'RandomForest — Top-20 Feature Importance'),
    (FIGS/'phase2_calibration_curve_valid.png', 'Phase-2 — Calibration (Valid)'),
    (FIGS/'phase2_f1_vs_threshold_valid.png', 'Phase-2 — F1 vs Threshold (Valid)'),
    (FIGS/'phase2_pr_curve_valid.png', 'Phase-2 — Precision–Recall (Valid)'),
    (FIGS/'phase2_cm_test_threshold_0.5.png', 'Phase-2 — CM (Test) thr=0.50'),
    (FIGS/'phase2_cm_test_threshold_0.22.png', 'Phase-2 — CM (Test) thr=0.22 (best on Valid)'),
]

for p, t in key_figs:
    show_if_exists(p, t)


## 6) اجرای دمو

### Streamlit
```bash
pip install streamlit
streamlit run app/streamlit_app.py
```

### CLI
```bash
python -m src.inference.predict_csv --input_csv data/raw/bank_marketing.csv --output_csv results/predictions_raw.csv
```
